# Notebook 3: Statistical Analysis
**Paper 5 – Cross-Framework Quantum Algorithm Benchmarking**

**Purpose**: Perform all statistical tests for RQ2–RQ5:
- RQ2: Pairwise JSD across frameworks (distribution equivalence)
- RQ4: Power-law scaling fitting (gate count vs qubit count)
- RQ5: QPack composite scores (S_runtime, S_accuracy, S_scalability, S_capacity, S_overall)

**Prerequisites**: Run notebooks 1 and 2 first.

**Outputs**:
- `benchmarks/metrics/statistical_tests.csv`
- `benchmarks/metrics/qpack_scores.csv`
- `benchmarks/metrics/power_law_fits.csv`

**Pipeline step**: Step 3 of 5

In [ ]:
import os, sys
QCANVAS_ROOT = os.path.abspath('../..')
if QCANVAS_ROOT not in sys.path:
    sys.path.insert(0, QCANVAS_ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from benchmarks.scripts.figure_styles import apply_paper_style, save_figure, FRAMEWORK_COLORS, FRAMEWORK_LABELS
from benchmarks.scripts.statistical_tests import (
    run_all_statistical_tests, fit_power_law,
    compute_runtime_score, compute_accuracy_score,
    compute_scalability_score, compute_capacity_score,
    compute_overall_score, JSD_DIVERGENT_THRESHOLD
)

In [ ]:
# ── RQ2: Pairwise JSD across all algorithms ──────────────────────────────────
distributions_dir = 'benchmarks/metrics/distributions'

print('Computing pairwise JSD for all algorithms @ 4096 shots...')
df_tests = run_all_statistical_tests(distributions_dir, shots=4096)
df_tests.to_csv('benchmarks/metrics/statistical_tests.csv', index=False)
print(f'Statistical tests complete: {len(df_tests)} rows')
print(f'\nAlgorithms with all-equivalent distributions (JSD < 0.01):')
print(df_tests[df_tests['all_equivalent']][['algorithm', 'n_qubits']].to_string(index=False))
print(f'\nAlgorithms with divergent distributions (JSD >= 0.05):')
div = df_tests[(
    (df_tests['jsd_qiskit_cirq']  >= JSD_DIVERGENT_THRESHOLD) |
    (df_tests['jsd_qiskit_pl']    >= JSD_DIVERGENT_THRESHOLD) |
    (df_tests['jsd_cirq_pl']      >= JSD_DIVERGENT_THRESHOLD)
)]
print(div[['algorithm', 'n_qubits', 'jsd_qiskit_cirq', 'jsd_qiskit_pl', 'jsd_cirq_pl']].to_string(index=False))

In [ ]:
# ── Fig. 9: Pairwise JSD heatmap ────────────────────────────────────────────
apply_paper_style()
from benchmarks.scripts.figure_styles import plot_jsd_heatmap

# Build heatmap DataFrame: rows=algorithm, cols=framework pairs
df_heat = df_tests.set_index('algorithm')[['jsd_qiskit_cirq', 'jsd_qiskit_pl', 'jsd_cirq_pl']]
df_heat.columns = ['Qiskit↔Cirq', 'Qiskit↔PennyLane', 'Cirq↔PennyLane']

fig, ax = plt.subplots(figsize=(8, len(df_heat) * 0.55 + 1.5))
plot_jsd_heatmap(ax, df_heat, title='Fig. 9 — Pairwise JSD Heatmap (4096 shots)')
plt.tight_layout()
save_figure(fig, 'fig09_jsd_heatmap', 'simulation')
plt.show()

In [ ]:
# ── RQ4: Power-law scaling fits ──────────────────────────────────────────────
df_struct = pd.read_csv('benchmarks/metrics/structural_metrics.csv')

variable_algos = ['ghz_state', 'deutsch_jozsa', 'grovers_algorithm', 'qrng', 'bernstein_vazirani']
power_law_rows = []

print('Power-law fits (gate count ~ N^a):' )
for fw in ['qiskit', 'cirq', 'pennylane']:
    for algo in variable_algos:
        sub = df_struct[(df_struct['algorithm'] == algo) & (df_struct['framework'] == fw)].sort_values('n_qubits')
        if len(sub) >= 3:
            fit = fit_power_law(sub['n_qubits'].tolist(), sub['total_gates'].tolist())
            print(f'  {fw:12s} | {algo:25s} → a={fit["a"]:5.3f}  R²={fit["r2"]:5.3f}  [{fit["scaling_class"]}]')
            power_law_rows.append({'algorithm': algo, 'framework': fw, **fit})

df_pl = pd.DataFrame(power_law_rows)
df_pl.to_csv('benchmarks/metrics/power_law_fits.csv', index=False)

In [ ]:
# ── Fig. 5: Gate count scaling for GHZ state ─────────────────────────────────
apply_paper_style()
from benchmarks.scripts.figure_styles import plot_scaling_lines

df_ghz = df_struct[df_struct['algorithm'] == 'ghz_state']
fig, ax = plt.subplots(figsize=(8, 5))
plot_scaling_lines(
    ax, df_ghz,
    x_col='n_qubits', y_col='total_gates',
    title='Fig. 5 — Gate Count Scaling: GHZ State (3–8 qubits)',
    ylabel='Total Gate Count',
)
plt.tight_layout()
save_figure(fig, 'fig05_ghz_scaling', 'structural')
plt.show()

In [ ]:
# ── QPack Composite Scores (RQ5) ─────────────────────────────────────────────
df_compile = pd.read_csv('benchmarks/metrics/compilation_times.csv')
df_compile = df_compile[df_compile['success']]

# Merge total_gates from structural metrics
df_merged = df_compile.merge(
    df_struct[['algorithm', 'framework', 'n_qubits', 'total_gates']],
    on=['algorithm', 'framework', 'n_qubits'], how='left'
)

runtime_scores = compute_runtime_score(df_merged)
print('S_compile_speed (QPack §IV-A):', runtime_scores)

In [ ]:
# ── Compute all QPack sub-scores and save ─────────────────────────────────────
scalability_scores = {}
for fw in ['qiskit', 'cirq', 'pennylane']:
    a_values = []
    for algo in variable_algos:
        row = df_pl[(df_pl['framework'] == fw) & (df_pl['algorithm'] == algo)]
        if not row.empty:
            a_values.append(row['a'].values[0])
    mean_a = float(np.nanmean(a_values)) if a_values else float('nan')
    scalability_scores[fw] = compute_scalability_score(mean_a)

print('S_scalability  (QPack §IV-C):', scalability_scores)

# Accuracy scores from JSD data
accuracy_scores = {}
for fw in ['qiskit', 'cirq', 'pennylane']:
    jsd_col = 'jsd_qiskit_cirq' if fw in ('qiskit', 'cirq') else 'jsd_qiskit_pl'
    jsd_map = dict(zip(df_tests['algorithm'], df_tests[jsd_col]))
    accuracy_scores[fw] = compute_accuracy_score(jsd_map, fw)
print('S_accuracy     (QPack §IV-B):', accuracy_scores)

# Capacity scores
capacity_scores = {}
for fw in ['qiskit', 'cirq', 'pennylane']:
    jsd_col = 'jsd_qiskit_cirq' if fw in ('qiskit', 'cirq') else 'jsd_qiskit_pl'
    jsd_by_n = dict(zip(df_tests['n_qubits'].astype(int), df_tests[jsd_col].astype(float)))
    capacity_scores[fw] = compute_capacity_score(jsd_by_n)
print('S_capacity     (QPack §IV-D):', capacity_scores)

# Overall scores
qpack_rows = []
for fw in ['qiskit', 'cirq', 'pennylane']:
    s_overall = compute_overall_score(
        S_speed=runtime_scores.get(fw, float('nan')),
        S_scale=scalability_scores.get(fw, float('nan')),
        S_acc=accuracy_scores.get(fw, float('nan')),
        S_cap=capacity_scores.get(fw, float('nan')),
    )
    qpack_rows.append({
        'framework': fw,
        'S_compile_speed': runtime_scores.get(fw),
        'S_scalability': scalability_scores.get(fw),
        'S_accuracy': accuracy_scores.get(fw),
        'S_capacity': capacity_scores.get(fw),
        'S_overall': s_overall,
    })

df_qpack = pd.DataFrame(qpack_rows)
df_qpack.to_csv('benchmarks/metrics/qpack_scores.csv', index=False)
print('\nQPack Scores Summary:')
print(df_qpack.round(3).to_string(index=False))
print('\nQPack ref: QuEST=183.2, Cirq=157.6, Qiskit Aer=147.2 (Donkers et al. 2022)')